# 05 — Analisis dengan Spark SQL (Bab 2.4a)Notebook ini mengimplementasikan analisis eksploratif menggunakan **Spark SQL** terhadap data transaksi ritel.

## 5.1 Inisialisasi & Load dari Parquet

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("05_SparkSQL_Analysis") \
    .master("local[*]") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

# Load dari Parquet (lebih cepat dari CSV)
df = spark.read.parquet("/output/retail_parquet")
df.createOrReplaceTempView("transactions")
print(f"✅ Loaded {df.count()} rows from Parquet")

✅ Loaded 1000 rows from Parquet


## 5.2 Query 1: Revenue per Kategori Produk

In [2]:
print("=== REVENUE PER KATEGORI PRODUK ===")
spark.sql("""
    SELECT 
        Product_Category,
        COUNT(*) as total_transaksi,
        SUM(Total_Amount) as total_revenue,
        ROUND(AVG(Total_Amount), 1) as avg_per_transaksi,
        SUM(Quantity) as total_unit_terjual
    FROM transactions
    GROUP BY Product_Category
    ORDER BY total_revenue DESC
""").show()

=== REVENUE PER KATEGORI PRODUK ===
+----------------+---------------+-------------+-----------------+------------------+
|Product_Category|total_transaksi|total_revenue|avg_per_transaksi|total_unit_terjual|
+----------------+---------------+-------------+-----------------+------------------+
|     Electronics|            342|       156905|            458.8|               849|
|        Clothing|            351|       155580|            443.2|               894|
|          Beauty|            307|       143515|            467.5|               771|
+----------------+---------------+-------------+-----------------+------------------+



## 5.3 Query 2: Distribusi Gender

In [3]:
print("=== DISTRIBUSI GENDER ===")
spark.sql("""
    SELECT 
        Gender,
        COUNT(*) as jumlah_pelanggan,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 1) as persentase,
        SUM(Total_Amount) as total_spending,
        ROUND(AVG(Total_Amount), 1) as avg_spending
    FROM transactions
    GROUP BY Gender
""").show()

=== DISTRIBUSI GENDER ===
+------+----------------+----------+--------------+------------+
|Gender|jumlah_pelanggan|persentase|total_spending|avg_spending|
+------+----------------+----------+--------------+------------+
|Female|             510|      51.0|        232840|       456.5|
|  Male|             490|      49.0|        223160|       455.4|
+------+----------------+----------+--------------+------------+



## 5.4 Query 3: Analisis per Kelompok Usia

In [4]:
print("=== SPENDING PER KELOMPOK USIA ===")
spark.sql("""
    SELECT 
        CASE 
            WHEN Age BETWEEN 18 AND 25 THEN '18-25 (Muda)'
            WHEN Age BETWEEN 26 AND 35 THEN '26-35 (Dewasa Muda)'
            WHEN Age BETWEEN 36 AND 45 THEN '36-45 (Dewasa)'
            WHEN Age BETWEEN 46 AND 55 THEN '46-55 (Dewasa Atas)'
            ELSE '56-64 (Pra-Lansia)'
        END as kelompok_usia,
        COUNT(*) as jumlah,
        SUM(Total_Amount) as total_spending,
        ROUND(AVG(Total_Amount), 1) as avg_spending,
        ROUND(AVG(Quantity), 1) as avg_quantity
    FROM transactions
    GROUP BY kelompok_usia
    ORDER BY total_spending DESC
""").show()

=== SPENDING PER KELOMPOK USIA ===
+-------------------+------+--------------+------------+------------+
|      kelompok_usia|jumlah|total_spending|avg_spending|avg_quantity|
+-------------------+------+--------------+------------+------------+
|46-55 (Dewasa Atas)|   229|        100690|       439.7|         2.6|
|26-35 (Dewasa Muda)|   205|         98480|       480.4|         2.6|
|     36-45 (Dewasa)|   202|         91870|       454.8|         2.5|
|       18-25 (Muda)|   169|         84550|       500.3|         2.4|
| 56-64 (Pra-Lansia)|   195|         80410|       412.4|         2.4|
+-------------------+------+--------------+------------+------------+



## 5.5 Query 4: Tren Penjualan Bulanan

In [5]:
print("=== TREN PENJUALAN BULANAN (2023) ===")
spark.sql("""
    SELECT 
        MONTH(Date) as bulan,
        COUNT(*) as total_transaksi,
        SUM(Total_Amount) as total_revenue,
        ROUND(AVG(Total_Amount), 1) as avg_per_transaksi
    FROM transactions
    WHERE YEAR(Date) = 2023
    GROUP BY MONTH(Date)
    ORDER BY bulan
""").show(12)

=== TREN PENJUALAN BULANAN (2023) ===
+-----+---------------+-------------+-----------------+
|bulan|total_transaksi|total_revenue|avg_per_transaksi|
+-----+---------------+-------------+-----------------+
|    1|             76|        35450|            466.4|
|    2|             85|        44060|            518.4|
|    3|             73|        28990|            397.1|
|    4|             86|        33870|            393.8|
|    5|            105|        53150|            506.2|
|    6|             77|        36715|            476.8|
|    7|             72|        35465|            492.6|
|    8|             94|        36960|            393.2|
|    9|             65|        23620|            363.4|
|   10|             96|        46580|            485.2|
|   11|             78|        34920|            447.7|
|   12|             91|        44690|            491.1|
+-----+---------------+-------------+-----------------+



## 5.6 Query 5: Top Spender per Kategori

In [6]:
print("=== TOP 5 PELANGGAN PER KATEGORI ===")
spark.sql("""
    SELECT * FROM (
        SELECT 
            Product_Category,
            Customer_ID,
            Total_Amount,
            ROW_NUMBER() OVER (PARTITION BY Product_Category ORDER BY Total_Amount DESC) as rank
        FROM transactions
    )
    WHERE rank <= 5
    ORDER BY Product_Category, rank
""").show(15)

=== TOP 5 PELANGGAN PER KATEGORI ===
+----------------+-----------+------------+----+
|Product_Category|Customer_ID|Total_Amount|rank|
+----------------+-----------+------------+----+
|          Beauty|    CUST074|        2000|   1|
|          Beauty|    CUST093|        2000|   2|
|          Beauty|    CUST139|        2000|   3|
|          Beauty|    CUST257|        2000|   4|
|          Beauty|    CUST281|        2000|   5|
|        Clothing|    CUST124|        2000|   1|
|        Clothing|    CUST166|        2000|   2|
|        Clothing|    CUST253|        2000|   3|
|        Clothing|    CUST269|        2000|   4|
|        Clothing|    CUST342|        2000|   5|
|     Electronics|    CUST015|        2000|   1|
|     Electronics|    CUST065|        2000|   2|
|     Electronics|    CUST072|        2000|   3|
|     Electronics|    CUST089|        2000|   4|
|     Electronics|    CUST109|        2000|   5|
+----------------+-----------+------------+----+



## 5.7 Query 6: Distribusi Harga

In [7]:
print("=== DISTRIBUSI PRICE TIER ===")
spark.sql("""
    SELECT 
        Price_per_Unit as harga,
        CASE
            WHEN Price_per_Unit IN (25, 30) THEN 'Budget'
            WHEN Price_per_Unit = 50 THEN 'Mid-Range'
            ELSE 'Premium'
        END as price_tier,
        COUNT(*) as jumlah_transaksi,
        SUM(Total_Amount) as total_revenue
    FROM transactions
    GROUP BY Price_per_Unit, price_tier
    ORDER BY Price_per_Unit
""").show()

=== DISTRIBUSI PRICE TIER ===
+-----+----------+----------------+-------------+
|harga|price_tier|jumlah_transaksi|total_revenue|
+-----+----------+----------------+-------------+
|   25|    Budget|             210|        13050|
|   30|    Budget|             183|        13350|
|   50| Mid-Range|             211|        26700|
|  300|   Premium|             197|       155400|
|  500|   Premium|             199|       247500|
+-----+----------+----------------+-------------+



## 5.8 Query 7: Cross-Tabulation Gender × Kategori

In [8]:
print("=== CROSS-TAB: GENDER × KATEGORI ===")
spark.sql("""
    SELECT 
        Gender,
        SUM(CASE WHEN Product_Category = 'Beauty' THEN Total_Amount ELSE 0 END) as Beauty,
        SUM(CASE WHEN Product_Category = 'Clothing' THEN Total_Amount ELSE 0 END) as Clothing,
        SUM(CASE WHEN Product_Category = 'Electronics' THEN Total_Amount ELSE 0 END) as Electronics,
        SUM(Total_Amount) as Grand_Total
    FROM transactions
    GROUP BY Gender
""").show()

=== CROSS-TAB: GENDER × KATEGORI ===
+------+------+--------+-----------+-----------+
|Gender|Beauty|Clothing|Electronics|Grand_Total|
+------+------+--------+-----------+-----------+
|Female| 74830|   81275|      76735|     232840|
|  Male| 68685|   74305|      80170|     223160|
+------+------+--------+-----------+-----------+

